### Lab 4.2: Batching and Regularization

In this lab you will learn how to set up a dataset to be processed in batches, rather than processing the entire dataset in each training iteration, and explore neural network regularization.

In [170]:
import numpy as np
import torch
from matplotlib import pyplot as plt

We will use the `diabetes` datasets from scikit-learn:

*Ten baseline variables, age, sex, body mass index, average blood pressure, and six blood serum measurements were obtained for each of n = 442 diabetes patients, as well as the response of interest, a quantitative measure of disease progression one year after baseline.*

This is a regression dataset, so we will use mean squared error as the loss function.

In [171]:
from sklearn.datasets import load_diabetes
  
diabetes = load_diabetes()
X = diabetes['data']
y = diabetes['target'][:,None]
X.shape, X.dtype, y.shape, y.dtype

((442, 10), dtype('float64'), (442, 1), dtype('float64'))

In [172]:
X.min(axis=0), X.max(axis=0)

(array([-0.10722563, -0.04464164, -0.0902753 , -0.1123988 , -0.12678067,
        -0.11561307, -0.10230705, -0.0763945 , -0.12609712, -0.13776723]),
 array([0.11072668, 0.05068012, 0.17055523, 0.13204362, 0.15391371,
        0.19878799, 0.18117906, 0.18523444, 0.13359728, 0.13561183]))

In [173]:
y.min(), y.max()

(np.float64(25.0), np.float64(346.0))

To make the learning algorithm work more smoothly, we we will subtract the mean of each feature.

Here `np.mean` calculates a mean, and `axis=0` tells NumPy to calculate the mean over the rows (calculate the mean of each column).

In [174]:
X -= np.mean(X,axis=0)
X /= np.std(X,axis=0)

# used clause to figure out why loss wasn't changing, found y wasn't normalized 
y -= np.mean(y)
y /= np.std(y)

Now we will convert our `X` and `y` arrays to torch Tensors.

In [175]:
X = torch.tensor(X).float()
y = torch.tensor(y).float()

### Exercises

1. Divide the data into train and test splits (`sklearn.model_selection.train_test_split`)
2. Create a neural network for this dataset.
3. Use `TensorDataset` and `DataLoader` to batch the dataset during training.  
4. Use `weight_decay` parameter to `optim.SGD` to introduce L2 regularization during training. Evaluate the effect of regularization on test set accuracy.

*Note: make sure to report the BEST test accuracy seen during training, not the last!*

In [176]:
# YOUR CODE GOES HERE
from sklearn import model_selection
from torch.utils.data import TensorDataset, DataLoader


X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y)

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32)

hidden_layer = 100
nn_model = torch.nn.Sequential(
    torch.nn.Linear(10, hidden_layer),
    torch.nn.ReLU(),
    torch.nn.Linear(hidden_layer, hidden_layer),
    torch.nn.ReLU(),
    torch.nn.Linear(hidden_layer, 1)
)

In [177]:
def train_model(model, train_dataloader, test_dataloader, weight_decay=1e-4):
    optim = torch.optim.SGD(model.parameters(), lr=1e-2, weight_decay=weight_decay)
    loss_fn = torch.nn.MSELoss()
    best_mse = float("inf")
    best_epoch = float("inf")

    for epoch in range(100):
        model.train()
    
        for batch_X, batch_y in train_dataloader:
            optim.zero_grad()
            outputs = model(batch_X)
            loss = loss_fn(outputs, batch_y)
            loss.backward()
            optim.step()

        model.eval()
        total_test_loss = 0
        with torch.no_grad():
            for batch_X, batch_y in test_dataloader:
                outputs = model(batch_X)
                loss = loss_fn(outputs, batch_y)
                total_test_loss += loss.item() * batch_X.size(0)
        test_mse = total_test_loss / len(test_dataloader.dataset)

        if test_mse < best_mse:
            best_mse = test_mse
            best_epoch = epoch

        print(f"Epoch {epoch}: test mse:{test_mse:.3f}, best mse:{best_mse:.3f} in epoch {best_epoch}")

In [178]:
train_model(nn_model, train_dataloader, test_dataloader) # with normalization

Epoch 0: test mse:0.856, best mse:0.856 in epoch 0
Epoch 1: test mse:0.730, best mse:0.730 in epoch 1
Epoch 2: test mse:0.646, best mse:0.646 in epoch 2
Epoch 3: test mse:0.590, best mse:0.590 in epoch 3
Epoch 4: test mse:0.559, best mse:0.559 in epoch 4
Epoch 5: test mse:0.523, best mse:0.523 in epoch 5
Epoch 6: test mse:0.505, best mse:0.505 in epoch 6
Epoch 7: test mse:0.495, best mse:0.495 in epoch 7
Epoch 8: test mse:0.484, best mse:0.484 in epoch 8
Epoch 9: test mse:0.474, best mse:0.474 in epoch 9
Epoch 10: test mse:0.471, best mse:0.471 in epoch 10
Epoch 11: test mse:0.473, best mse:0.471 in epoch 10
Epoch 12: test mse:0.471, best mse:0.471 in epoch 12
Epoch 13: test mse:0.463, best mse:0.463 in epoch 13
Epoch 14: test mse:0.462, best mse:0.462 in epoch 14
Epoch 15: test mse:0.468, best mse:0.462 in epoch 14
Epoch 16: test mse:0.466, best mse:0.462 in epoch 14
Epoch 17: test mse:0.476, best mse:0.462 in epoch 14
Epoch 18: test mse:0.471, best mse:0.462 in epoch 14
Epoch 19: tes

In [179]:
train_model(nn_model, train_dataloader, test_dataloader, weight_decay=0) # no normalization

Epoch 0: test mse:0.502, best mse:0.502 in epoch 0
Epoch 1: test mse:0.516, best mse:0.502 in epoch 0
Epoch 2: test mse:0.516, best mse:0.502 in epoch 0
Epoch 3: test mse:0.500, best mse:0.500 in epoch 3
Epoch 4: test mse:0.500, best mse:0.500 in epoch 4
Epoch 5: test mse:0.507, best mse:0.500 in epoch 4
Epoch 6: test mse:0.518, best mse:0.500 in epoch 4
Epoch 7: test mse:0.514, best mse:0.500 in epoch 4
Epoch 8: test mse:0.511, best mse:0.500 in epoch 4
Epoch 9: test mse:0.525, best mse:0.500 in epoch 4
Epoch 10: test mse:0.514, best mse:0.500 in epoch 4
Epoch 11: test mse:0.522, best mse:0.500 in epoch 4
Epoch 12: test mse:0.523, best mse:0.500 in epoch 4
Epoch 13: test mse:0.517, best mse:0.500 in epoch 4
Epoch 14: test mse:0.508, best mse:0.500 in epoch 4
Epoch 15: test mse:0.514, best mse:0.500 in epoch 4
Epoch 16: test mse:0.519, best mse:0.500 in epoch 4
Epoch 17: test mse:0.530, best mse:0.500 in epoch 4
Epoch 18: test mse:0.517, best mse:0.500 in epoch 4
Epoch 19: test mse:0.5

When using L2 normalization with a weight decay of 0.0001 the test set received the best mean square error of 0.462 in epoch 14. Without L2 normalization, the test set achieved a best mean squared error of 0.5 in epoch 4. This demonstrates how using normalization when training can can generalize better, and therefore perform better on the test set. 